In [1]:
import os
os.chdir('../')
%pwd

'/Users/kiranprasadjp/Documents/Pros/HHMI_Janelia_AI_Engineer'

In [13]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    scale: str

In [14]:
from src.task2 import logger
from src.task2.constants import *
from src.task2.utils.common import read_yaml, create_directories

In [15]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir= Path(config.root_dir),
            source_URL= str(config.source_URL),
            local_data_file=Path(config.local_data_file),
            scale= str(self.params.scale)
        )

        return data_ingestion_config

In [16]:
import os
import numpy as np
import fibsem_tools as fst
from pathlib import Path
from src.task2 import logger
from src.task2.utils.common import get_size

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            local_file_path= self.config.local_data_file
            scale=self.config.scale

            os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
            # 2. Check if file already exists to save time/bandwidth
            if not os.path.exists(local_file_path):
                logger.info(f"Connecting to s3 {dataset_url}...")

                tree = fst.read_xarray(dataset_url, storage_options={'anon': True})

                data_array = tree[scale].data
                logger.info(f"Fetching chunks from S3 {scale}...")
                # Based on your metadata chunksize of 448
                z_range = slice(1000, 1448)       # 2 vertical chunks
                y_range = slice(2240, 2688)   # 1 chunk wide
                x_range = slice(2240, 2688)   # 1 chunk deep

                # .compute() pulls the data from S3 into memory
                volume = data_array[z_range, y_range, x_range].compute()
                # 5. Save as .npy for fast loading into DINO v3
                np.save(local_file_path, volume)
                logger.info(f"Download complete! Volume {volume.shape} saved to: {local_file_path} (Size: {get_size(Path(local_file_path))})")
            else:
                logger.info(f"File already exists at: {local_file_path} (Size: {get_size(Path(local_file_path))})")
        except Exception as e:
            logger.error(f"Error occurred while downloading: {e}")
            raise e

In [18]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
except Exception as e:
    raise e

[2026-04-01 21:04:32,302: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-01 21:04:32,304: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-01 21:04:32,304: INFO: common: created directory at: artifacts]
[2026-04-01 21:04:32,305: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-01 21:04:32,305: INFO: 3382771004: Connecting to s3 s3://janelia-cosem-datasets/jrc_mus-liver/jrc_mus-liver.n5/em/fibsem-uint8/...]
[2026-04-01 21:04:39,709: INFO: 3382771004: Fetching chunks from S3...]
[2026-04-01 21:04:54,430: INFO: 3382771004: Download complete! Volume (448, 448, 448) saved to: artifacts/data_ingestion/liver_chunk.npy (Size: ~ 87808 KB)]


# Trails

In [12]:
import quilt3 as q3
b = q3.Bucket("s3://janelia-cosem-datasets")
b.ls("jrc_mus-liver/jrc_mus-liver.zarr/")

([{'Prefix': 'jrc_mus-liver/jrc_mus-liver.zarr/recon-1/'}],
 [{'ETag': '"99914b932bd37a50b983c5e7c90ae93b"',
   'Size': 2,
   'StorageClass': 'STANDARD',
   'Key': 'jrc_mus-liver/jrc_mus-liver.zarr/.zattrs',
   'VersionId': 'null',
   'IsLatest': True,
   'LastModified': datetime.datetime(2024, 8, 1, 3, 42, 4, tzinfo=tzutc()),
   'Owner': {'ID': '00987910cca20cc0c9915bc698607c0e7dee9d16710c48e2d14a20f7a05d4fe5'}},
  {'ETag': '"e20297935e73dd0154104d4ea53040ab"',
   'Size': 24,
   'StorageClass': 'STANDARD',
   'Key': 'jrc_mus-liver/jrc_mus-liver.zarr/.zgroup',
   'VersionId': 'null',
   'IsLatest': True,
   'LastModified': datetime.datetime(2024, 8, 1, 3, 42, 4, tzinfo=tzutc()),
   'Owner': {'ID': '00987910cca20cc0c9915bc698607c0e7dee9d16710c48e2d14a20f7a05d4fe5'}}],
 [])

# From Fibsem-tools GitHub

In [22]:
from fibsem_tools import read_xarray, read
from rich import print # pretty printing
creds = {'anon': True} # anonymous credentials for s3
url = 's3://janelia-cosem-datasets/jrc_sum159-1/jrc_sum159-1.n5/em/fibsem-uint16/' # path to a group of arrays on s3
group = read(url, storage_options=creds) # this returns a zarr group, which in this case is a collection of arrays
print(tuple(group.arrays())) # this shows all the arrays in the group
# (
#     ('s0', <zarr.core.Array '/em/fibsem-uint16/s0' (7632, 2800, 16000) uint16 read-only>),
#     ('s1', <zarr.core.Array '/em/fibsem-uint16/s1' (3816, 1400, 8000) uint16 read-only>),
#     ('s2', <zarr.core.Array '/em/fibsem-uint16/s2' (1908, 700, 4000) uint16 read-only>),
#     ('s3', <zarr.core.Array '/em/fibsem-uint16/s3' (954, 350, 2000) uint16 read-only>),
#     ('s4', <zarr.core.Array '/em/fibsem-uint16/s4' (477, 175, 1000) uint16 read-only>),
#     ('s5', <zarr.core.Array '/em/fibsem-uint16/s5' (239, 88, 500) uint16 read-only>)
# )
tree = read_xarray(url, storage_options=creds) # read the group as a DataTree, a collection of xarray objects
print(tree)

/Users/kiranprasadjp/Documents/Pros/HHMI_Janelia_AI_Engineer/.venv/lib/python3.13/site-packages/fibsem_tools/io/n5/core.py:101: FutureWarning: The N5FSStore is deprecated and will be removed in a Zarr-Python version 3, see https://github.com/zarr-developers/zarr-python/issues/1274 and https://github.com/zarr-developers/n5py for more information.
  store = DEFAULT_N5_STORE(store, **kwargs.pop("storage_options", {}))


(
    ('s0', <zarr.core.Array '/em/fibsem-uint16/s0' (7632, 2800, 16000) uint16 read-only>),
    ('s1', <zarr.core.Array '/em/fibsem-uint16/s1' (3816, 1400, 8000) uint16 read-only>),
    ('s2', <zarr.core.Array '/em/fibsem-uint16/s2' (1908, 700, 4000) uint16 read-only>),
    ('s3', <zarr.core.Array '/em/fibsem-uint16/s3' (954, 350, 2000) uint16 read-only>),
    ('s4', <zarr.core.Array '/em/fibsem-uint16/s4' (477, 175, 1000) uint16 read-only>),
    ('s5', <zarr.core.Array '/em/fibsem-uint16/s5' (239, 88, 500) uint16 read-only>)
)

<xarray.DataTree 'fibsem-uint16'>
Group: /
│   Attributes:
│       axes:             ['x', 'y', 'z']
│       multiscales:      [{'datasets': [{'path': 's0', 'transform': {'axes': ['z...
│       pixelResolution:  {'dimensions': [4.0, 4.0, 4.56], 'unit': 'nm'}
│       scales:           [[1, 1, 1], [2, 2, 2], [4, 4, 4], [8, 8, 8], [16, 16, 1...
│       units:            ['nm', 'nm', 'nm']
├── Group: /s0
│       Dimensions:  (z: 7632, y: 2800, x: 16000)
│       Coordinates:
│         * z        (z) float64 61kB 0.0 4.56 9.12 ... 3.479e+04 3.479e+04 3.48e+04
│         * y        (y) float64 22kB 0.0 4.0 8.0 12.0 ... 1.119e+04 1.119e+04 1.12e+04
│         * x        (x) float64 128kB 0.0 4.0 8.0 12.0 ... 6.399e+04 6.399e+04 6.4e+04
│       Data variables:
│           data     (z, y, x) uint16 684GB dask.array<chunksize=(384, 384, 384), meta=np.ndarray>
├── Group: /s1
│       Dimensions:  (z: 3816, y: 1400, x: 8000)
│       Coordinates:
│         * z        (z) float64 31kB 2.28 11.4 20.52 ... 3.478e+04 3.479e+04 3.48e+04
│         * y        (y) float64 11kB 2.0 10.0 18.0 ... 1.118e+04 1.119e+04 1.119e+04
│         * x        (x) float64 64kB 2.0 10.0 18.0 ... 6.398e+04 6.399e+04 6.399e+04
│       Data variables:
│           data     (z, y, x) uint16 85GB dask.array<chunksize=(384, 384, 384), meta=np.ndarray>
├── Group: /s2
│       Dimensions:  (z: 1908, y: 700, x: 4000)
│       Coordinates:
│         * z        (z) float64 15kB 6.84 25.08 43.32 ... 3.475e+04 3.477e+04 3.479e+04
│         * y        (y) float64 6kB 6.0 22.0 38.0 ... 1.116e+04 1.117e+04 1.119e+04
│         * x        (x) float64 32kB 6.0 22.0 38.0 ... 6.396e+04 6.397e+04 6.399e+04
│       Data variables:
│           data     (z, y, x) uint16 11GB dask.array<chunksize=(384, 384, 384), meta=np.ndarray>
├── Group: /s3
│       Dimensions:  (z: 954, y: 350, x: 2000)
│       Coordinates:
│         * z        (z) float64 8kB 15.96 52.44 88.92 ... 3.471e+04 3.474e+04 3.478e+04
│         * y        (y) float64 3kB 14.0 46.0 78.0 ... 1.112e+04 1.115e+04 1.118e+04
│         * x        (x) float64 16kB 14.0 46.0 78.0 ... 6.392e+04 6.395e+04 6.398e+04
│       Data variables:
│           data     (z, y, x) uint16 1GB dask.array<chunksize=(384, 350, 384), meta=np.ndarray>
├── Group: /s4
│       Dimensions:  (z: 477, y: 175, x: 1000)
│       Coordinates:
│         * z        (z) float64 4kB 34.2 107.2 180.1 ... 3.462e+04 3.469e+04 3.476e+04
│         * y        (y) float64 1kB 30.0 94.0 158.0 ... 1.104e+04 1.11e+04 1.117e+04
│         * x        (x) float64 8kB 30.0 94.0 158.0 ... 6.384e+04 6.39e+04 6.397e+04
│       Data variables:
│           data     (z, y, x) uint16 167MB dask.array<chunksize=(477, 175, 768), meta=np.ndarray>
└── Group: /s5
        Dimensions:  (z: 239, y: 88, x: 500)
        Coordinates:
          * z        (z) float64 2kB 70.68 216.6 362.5 ... 3.451e+04 3.465e+04 3.48e+04
          * y        (y) float64 704B 62.0 190.0 318.0 ... 1.094e+04 1.107e+04 1.12e+04
          * x        (x) float64 4kB 62.0 190.0 318.0 ... 6.368e+04 6.381e+04 6.393e+04
        Data variables:
            data     (z, y, x) uint16 21MB dask.array<chunksize=(239, 88, 500), meta=np.ndarray>

In [12]:

url = 's3://janelia-cosem-datasets/jrc_mus-liver/jrc_mus-liver.n5/em/fibsem-uint8/'

tree = fst.read_xarray(url, storage_options={'anon': True})

In [10]:
tree

<xarray.DataArray 'array-b697458d23884a8c3319e37bafddfd57' (z: 4466, y: 6364,
                                                            x: 6373)> Size: 181GB
dask.array<array, shape=(4466, 6364, 6373), dtype=uint8, chunksize=(448, 448, 448), chunktype=numpy.ndarray>
Coordinates:
  * z        (z) float64 36kB 4.0 20.0 36.0 ... 7.141e+04 7.143e+04 7.144e+04
  * y        (y) float64 51kB 4.0 20.0 36.0 ... 1.018e+05 1.018e+05 1.018e+05
  * x        (x) float64 51kB 4.0 20.0 36.0 ... 1.019e+05 1.019e+05 1.02e+05
Attributes:
    pixelResolution:  {'dimensions': [16.0, 16.0, 16.0], 'unit': 'nm'}
    transform:        {'axes': ['z', 'y', 'x'], 'order': 'C', 'scale': [16.0,...

In [ ]:
import fibsem_tools as fst

s1_data = tree['s1'].data
small_crop = s1_data[1000:1448, 2240:2688, 2240:2688].compute()

print(f"Downloaded Shape: {small_crop.shape}")
print(f"Data Size: {small_crop.nbytes / (1024**2):.2f} MB")


Downloaded Shape: (448, 448, 448)
Data Size: 85.75 MB
